# 📖 Notebook 3: Social Graph Storage

The social graph is the backbone of any social network. It answers questions like:  
"Who does Alice follow?", "Who follows Alice?", and "Do Alice and Bob follow each other?"

## Learning Objectives

By the end of this notebook, you'll understand:
- How to model uni-directional follow relationships in a relational database
- The difference between adjacency lists and adjacency matrices
- How to query the social graph efficiently with indexes
- When you'd need a dedicated graph database (and when you don't)

## 🛠️ Setup

```bash
cd system-designs/fb-news-feed
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsfeed_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM follows")
    print(f"✅ Connected — {cur.fetchone()[0]} follow relationships")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 What Is a Social Graph?

A social graph is a **directed graph** where:
- Each **node** is a user
- Each **edge** is a follow relationship (A follows B)

```
    Alice ──follows──► Bob
      │                 │
      │              follows
   follows              │
      │                 ▼
      ▼               Carol
    Celebrity
```

Key difference from "friends":
- **Follow** = uni-directional (Alice follows Bob, but Bob may not follow Alice)
- **Friend** = bi-directional (both must agree)

Facebook started with friends, but later added follows.  
Twitter, Instagram, TikTok all use follows.

## Storage Option 1: Relational Table (What We Use)

The simplest approach — store edges as rows in a table:

```sql
CREATE TABLE follows (
    follower_id INTEGER,  -- who is doing the following
    followee_id INTEGER,  -- who is being followed
    PRIMARY KEY (follower_id, followee_id)
);
```

This is an **adjacency list** stored in a relational table.

In [ ]:
# Let's explore our social graph

conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Who does user 1 follow?
cur.execute("""
    SELECT u.id, u.display_name
    FROM follows f
    JOIN users u ON u.id = f.followee_id
    WHERE f.follower_id = 1
    ORDER BY u.id
""")
followees = cur.fetchall()

print(f"👤 User 1 follows {len(followees)} users:")
for u in followees:
    print(f"   → {u['display_name']} (id={u['id']})")

print()

# Who follows celebrity Alice (id=51)?
cur.execute("""
    SELECT COUNT(*) as cnt
    FROM follows
    WHERE followee_id = 51
""")
print(f"🌟 Celebrity Alice has {cur.fetchone()['cnt']} followers")

conn.close()

## The Three Essential Queries

For a News Feed, we need these three queries to be fast:

| Query | Purpose | Used By |
|-------|---------|--------|
| Who does A follow? | Build A's feed | Feed Service |
| Who follows A? | Fan-out on write | Post Service |
| Does A follow B? | Check relationship | Follow button UI |

In [ ]:
def who_does_user_follow(user_id: int) -> list:
    """Query 1: Get all users that user_id follows."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    result = [row[0] for row in cur.fetchall()]
    conn.close()
    return result


def who_follows_user(user_id: int) -> list:
    """Query 2: Get all followers of user_id."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT follower_id FROM follows WHERE followee_id = %s",
        (user_id,)
    )
    result = [row[0] for row in cur.fetchall()]
    conn.close()
    return result


def does_a_follow_b(a: int, b: int) -> bool:
    """Query 3: Check if user A follows user B."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "SELECT 1 FROM follows WHERE follower_id = %s AND followee_id = %s",
        (a, b)
    )
    result = cur.fetchone() is not None
    conn.close()
    return result


# Demo all three queries
print("📊 Query 1 — Who does user 1 follow?")
followees = who_does_user_follow(1)
print(f"   {followees}\n")

print("📊 Query 2 — Who follows celebrity 51?")
followers = who_follows_user(51)
print(f"   {len(followers)} followers: {followers[:10]}...\n")

print("📊 Query 3 — Does user 1 follow user 51?")
print(f"   {does_a_follow_b(1, 51)}")
print(f"   Does user 1 follow user 99? {does_a_follow_b(1, 99)}")

## ❌ Bad → ✅ Best: What Happens Without an Index?

Before we celebrate our indexes, let's see what a **bad** design looks like.
We'll temporarily drop the reverse index and run `EXPLAIN ANALYZE` to see
PostgreSQL fall back to a **sequential scan** — i.e. reading every row in
the table. At small scale it's still fast; at 2 billion rows it's fatal.


In [ ]:
conn = get_db()
cur = conn.cursor()

# ❌ BAD: drop the reverse index and run the "who follows user 51" query
cur.execute("DROP INDEX IF EXISTS idx_follows_followee")
cur.execute("ANALYZE follows")  # refresh stats so the planner picks wisely

cur.execute("EXPLAIN ANALYZE SELECT follower_id FROM follows WHERE followee_id = 51")
print("❌ Without index (expect 'Seq Scan'):")
for row in cur.fetchall():
    print("   " + row[0])

# ✅ BEST: re-create the index and run the same query
cur.execute("CREATE INDEX idx_follows_followee ON follows(followee_id)")
cur.execute("ANALYZE follows")

print()
cur.execute("EXPLAIN ANALYZE SELECT follower_id FROM follows WHERE followee_id = 51")
print("✅ With index (expect 'Index Scan' / 'Bitmap Index Scan'):")
for row in cur.fetchall():
    print("   " + row[0])

conn.commit()
conn.close()
print()
print("💡 Notice 'Seq Scan' (bad) vs 'Bitmap/Index Scan' (best).")
print("   With 504 rows the difference is tiny; with 500 billion it is")
print("   the difference between a working system and a melted server.")


## Why Indexes Matter

Our `follows` table has two important indexes:

```sql
PRIMARY KEY (follower_id, followee_id)  -- fast lookup by follower
INDEX idx_follows_followee (followee_id) -- fast lookup by followee
```

Without these indexes, every query would scan the entire table.  
Let's see the difference.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Show the execution plan for each query
queries = [
    ("Who does user 1 follow?",
     "EXPLAIN ANALYZE SELECT followee_id FROM follows WHERE follower_id = 1"),
    ("Who follows user 51?",
     "EXPLAIN ANALYZE SELECT follower_id FROM follows WHERE followee_id = 51"),
    ("Does 1 follow 51?",
     "EXPLAIN ANALYZE SELECT 1 FROM follows WHERE follower_id = 1 AND followee_id = 51"),
]

for label, query in queries:
    cur.execute(query)
    plan = cur.fetchall()
    print(f"📋 {label}")
    for row in plan:
        line = row[0]
        # Highlight important parts
        if "Index" in line or "Seq Scan" in line or "Execution Time" in line:
            print(f"   ► {line}")
    print()

conn.close()
print("💡 All queries use Index Scan — no slow sequential scans!")
print("   The PRIMARY KEY covers queries by follower_id.")
print("   The idx_follows_followee index covers queries by followee_id.")

## Bonus: Mutual Follows and Suggestions

With our simple table, we can answer more complex social graph questions.

In [ ]:
def mutual_follows(user_a: int, user_b: int) -> bool:
    """Check if two users follow each other (mutual/friends)."""
    return does_a_follow_b(user_a, user_b) and does_a_follow_b(user_b, user_a)


def suggest_follows(user_id: int, limit: int = 5) -> list:
    """
    Suggest users to follow based on "friends of friends".
    
    Logic: Find users that people-I-follow also follow,
    but I don't follow yet. Rank by how many of my followees
    also follow them.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    cur.execute("""
        SELECT u.id, u.display_name, COUNT(*) as mutual_count
        FROM follows f1
        JOIN follows f2 ON f2.follower_id = f1.followee_id
        JOIN users u ON u.id = f2.followee_id
        WHERE f1.follower_id = %s
          AND f2.followee_id != %s
          AND f2.followee_id NOT IN (
              SELECT followee_id FROM follows WHERE follower_id = %s
          )
        GROUP BY u.id, u.display_name
        ORDER BY mutual_count DESC
        LIMIT %s
    """, (user_id, user_id, user_id, limit))
    
    suggestions = cur.fetchall()
    conn.close()
    return suggestions


# Demo: mutual follows
conn = get_db()
cur = conn.cursor()
cur.execute("SELECT follower_id, followee_id FROM follows WHERE follower_id <= 5 AND followee_id <= 5 ORDER BY 1, 2")
edges = cur.fetchall()
conn.close()

print("🔗 Follow edges among users 1–5:")
for a, b in edges:
    mutual = "(mutual ✅)" if (b, a) in edges else ""
    print(f"   User {a} → User {b} {mutual}")

print()

# Demo: follow suggestions
suggestions = suggest_follows(user_id=1, limit=5)
print("💡 Follow suggestions for user 1 (friends-of-friends):")
for s in suggestions:
    print(f"   {s['display_name']} — {s['mutual_count']} mutual connections")

## Follow / Unfollow Operations

In [ ]:
def follow_user(follower_id: int, followee_id: int) -> str:
    """
    Create a follow relationship.
    Uses INSERT ... ON CONFLICT DO NOTHING to make it idempotent
    (clicking "follow" twice won't cause an error).
    """
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO follows (follower_id, followee_id) VALUES (%s, %s) ON CONFLICT DO NOTHING",
        (follower_id, followee_id)
    )
    was_new = cur.rowcount > 0
    conn.commit()
    conn.close()
    return "followed" if was_new else "already following"


def unfollow_user(follower_id: int, followee_id: int) -> str:
    """Remove a follow relationship."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute(
        "DELETE FROM follows WHERE follower_id = %s AND followee_id = %s",
        (follower_id, followee_id)
    )
    was_removed = cur.rowcount > 0
    conn.commit()
    conn.close()
    return "unfollowed" if was_removed else "was not following"


# Demo
print("Does user 1 follow user 50?", does_a_follow_b(1, 50))
print("Action:", follow_user(1, 50))
print("Does user 1 follow user 50 now?", does_a_follow_b(1, 50))
print("Follow again (idempotent):", follow_user(1, 50))
print("Unfollow:", unfollow_user(1, 50))
print("Does user 1 follow user 50 now?", does_a_follow_b(1, 50))

## Graph Statistics

Understanding the shape of your graph helps make scaling decisions.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Total edges
cur.execute("SELECT COUNT(*) FROM follows")
total_edges = cur.fetchone()[0]

# Total users
cur.execute("SELECT COUNT(*) FROM users")
total_users = cur.fetchone()[0]

# Follower distribution
cur.execute("""
    SELECT 
        MIN(cnt) as min_followers,
        AVG(cnt)::int as avg_followers,
        MAX(cnt) as max_followers
    FROM (
        SELECT followee_id, COUNT(*) as cnt
        FROM follows
        GROUP BY followee_id
    ) sub
""")
fmin, favg, fmax = cur.fetchone()

# Following distribution
cur.execute("""
    SELECT 
        MIN(cnt) as min_following,
        AVG(cnt)::int as avg_following,
        MAX(cnt) as max_following
    FROM (
        SELECT follower_id, COUNT(*) as cnt
        FROM follows
        GROUP BY follower_id
    ) sub
""")
gmin, gavg, gmax = cur.fetchone()

conn.close()

print("📊 Social Graph Statistics")
print("=" * 40)
print(f"  Users:          {total_users}")
print(f"  Edges (follows): {total_edges}")
print(f"  Avg edges/user: {total_edges / total_users:.1f}")
print()
print(f"  Followers:  min={fmin}, avg={favg}, max={fmax}")
print(f"  Following:  min={gmin}, avg={gavg}, max={gmax}")
print()
print("💡 The max follower count is much higher than the max following count.")
print("   This asymmetry (celebrities) is what makes fan-out on write tricky.")

## When Do You Need a Graph Database?

| Query Type | Relational DB | Graph DB (Neo4j) |
|------------|:---:|:---:|
| Who does A follow? | ✅ Fast | ✅ Fast |
| Who follows A? | ✅ Fast (with index) | ✅ Fast |
| Does A follow B? | ✅ Fast (PK lookup) | ✅ Fast |
| Friends-of-friends | ⚠️ OK (1 JOIN) | ✅ Fast |
| 3+ hops (A→B→C→D) | ❌ Slow (nested JOINs) | ✅ Fast |
| Shortest path | ❌ Very slow | ✅ Built-in |
| Recommendation engine | ❌ Not practical | ✅ Designed for this |

**For a News Feed**, a relational table is perfectly fine.  
You only need 1-hop queries (direct follows). No traversals needed.

**Use a graph DB** when you need multi-hop traversals:  
social recommendations, fraud detection, knowledge graphs.

## 📚 Summary

### Key Takeaways

1. **Follows** are directed edges in a graph — store them as rows in a table
2. You need **two indexes**: one by follower (PK), one by followee
3. A relational DB handles 1-hop social graph queries efficiently
4. **Friends-of-friends** (2-hop) suggestions work with a single JOIN
5. Only use a graph database when you need multi-hop traversals

### Interview Tips

- Mention that Facebook uses both relational and graph storage depending on the query
- Explain why you chose a relational table: "we only need 1-hop lookups for the feed"
- The composite primary key `(follower_id, followee_id)` makes follow idempotent
- DynamoDB's Global Secondary Index (GSI) serves the same purpose as our reverse index

### Next Up

In **Notebook 4**, we'll explore **Feed Caching Strategies** — using Redis to cache feeds and handle hot keys.